Notebook to test our model on real data. This is a kind of test set.  


I start with the most obvious sample categories:
- clusters of trash
- Scattered litter
- Less litter, scattered
- hawkers
- flowers
- rocks

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import show, DiskImage, DiskBooleanMask, overlay_mask_on_img as OV
from mtrain.example_dir import ExampleDir
from tqdm import tqdm

In [ ]:
DS = Path("/Users/hariomnarang/Desktop/personal/roads/datasets")
DELHI_SAMPLES_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test"
)
TEST_SET_PATH = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/test_set"
)

In [ ]:
LOTS_OF_LITTER = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/532628267735038",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/271937142353543",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/1407390010751492",
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/samples_mapillary/100/26210846388517270"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/933090527327250"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/904411971827702"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/409495874297166"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/471611377923092"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/1132072153938604"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/318666179638135"
    ),
]

BROKEN_ROAD = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/1765386860738037",
]

SCATTERED = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/167994091824213",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/671811303998480",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/173239464669959",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/125035310018047",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/1634936550227999",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/291967965907673",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/1384373959095157",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/196244582323540",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/801754743801999",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/822560908385930",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/299603061792440",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/1397649970633008",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/2985603351766082",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/144348374637763",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/843858569843204",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/332198098541859",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/3625083887603287",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/2769241939938889",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/669848760702786",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/955227375338093",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/212281790352108",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/214125100125056",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/1490763574593712",
]

BLUR = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/161598622858285",
]

FLOWERS = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/2800723576860190",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/871543466734475",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/2562170430756718",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/2926084337660344",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/1043236500247070",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/930842434423691",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/550822005934553",
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/935180533980655"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/1019440569393347"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/1217095282865379"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/857578969238303"
    ),
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/1253130721794975",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/831014837763053",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/322214212632654",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/3799706853575740",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/369602612275380",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/1000795241832418",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/1357433225213341",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/1131950611202217",
]


STONES = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/566935955515605",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/195067892836607",
]

WALLS = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/884772330133126",
]

WEIRD = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/8792094944191192",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/810183961580401",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/672731224796321",
]

WIDE_ANGLE = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/992742956405587",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/535408762984577",
]

ALL_DIRS = (
    WIDE_ANGLE
    + WEIRD
    + WALLS
    + STONES
    + FLOWERS
    + BLUR
    + SCATTERED
    + BROKEN_ROAD
    + LOTS_OF_LITTER
)

# add samples to test set

In [ ]:
def get_image_dirs(samples_dir):
    for d in samples_dir.glob("*"):
        if not d.is_dir() or not (d / "image.jpg").exists():
            continue
        yield d


it = get_image_dirs(DELHI_SAMPLES_DIR)


In [ ]:
p = next(it)
print(p)
show([plt.imread(p / "image.jpg")], ncols=1)

## move samples to test set

In [ ]:
from mtrain.example_dir import create_dirs_for_images, ExampleDir, run_bulk_inference

In [ ]:
import shutil

for d in ALL_DIRS:
    dest = TEST_SET_PATH / Path(d).name
    if not dest.exists():
        shutil.copytree(d, dest)

# run bulk inference

In [ ]:
# force run
dirs = [d for d in TEST_SET_PATH.glob("*") if d.is_dir()]
for d in tqdm(dirs):
    edir = ExampleDir(d)
    edir.trash_probs_path(True)
    edir.other_probs_path(True)
    edir.flower_neg_probs_path(True)
    edir.flower_pos_probs_path(True)

# Visualize results

In [ ]:
dirs = [d for d in TEST_SET_PATH.glob("*") if d.is_dir()]
len(dirs)

In [ ]:
idx = 0

In [ ]:
def get_areas(binary_mask):
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary_mask, connectivity=8
    )

    areas = []
    lengths = []
    for label in range(1, num_labels):  # skip label 0 (background)
        areas.append(stats[label, cv2.CC_STAT_AREA])
        lengths.append(stats[label, cv2.CC_STAT_HEIGHT])


    return areas, lengths

In [ ]:
from mtrain.example_dir.core import (
    get_default_negmask_learner,
    NEG_MASK_UNBLURRED_MODEL_PATH,
    NEGMASK_SIZE,
    NEG_MASK_STEP_EDGE_MODEL_PATH,
)
from mtrain.neg_mask.model.predict import trash as pred_trash

unblurred_learner = get_default_negmask_learner(NEG_MASK_UNBLURRED_MODEL_PATH)
step_edge_learner = get_default_negmask_learner(NEG_MASK_STEP_EDGE_MODEL_PATH)

In [ ]:
idx = 0
all_areas, all_lengths = [], []

In [ ]:
# idx -= 1
from mtrain.example_dir.core import NEGMASK_BLUR_KERNEL_SZ, NEGMASK_BLUR_KERNEL_SIGMA, NEGMASK_BBOX_PAD, NEGMASK_STEP_EDGE_MAX_NOISE
from mtrain.utils import draw_grid_cv2

edir = ExampleDir(dirs[idx])


orig = DiskBooleanMask.load(edir.trimmed_mask_path())
image = DiskImage.load(edir.image_path)

trash_probs = np.load(edir.trash_probs_path())
other_probs = np.load(edir.other_probs_path())

fpos_probs = np.load(edir.flower_pos_probs_path())
fneg_probs = np.load(edir.flower_neg_probs_path())


unmask = orig & (trash_probs > other_probs)
flower_mask = orig & (fpos_probs > fneg_probs)
unmask = unmask & (fneg_probs > fpos_probs)


# bool2 is subset of bool1
# i need stuff not in bool2 but in bool1
# not_in_flower = unmask & (~with_flower)


areas, lengths = get_areas(unmask)

print("Areas", areas)
print("lenghts", lengths)

all_areas.extend(areas)
all_lengths.extend(lengths)
show(
    [
        image,
        OV(image, unmask),
        # OV(image, flower_mask.astype(bool)),
        # OV(image, orig.astype(bool)),
        # image,
        # OV(image, semask.astype(bool)),
    ],
    (20,20),
    2,
)

idx += 1

In [ ]:
all_widths = [ar / le for (ar,le) in zip(all_areas, all_lengths)]

In [ ]:
# stats = [(a,l,w) for (a,l,w) in zip(all_areas, all_lengths, all_widths)]
lenar = np.array(all_lengths)
widar = np.array(all_widths)

print(lenar.mean(), widar.mean())
print(lenar.max(), widar.max())
print(np.percentile(lenar, 50), np.percentile(widar, 50))
print(np.percentile(lenar, 25), np.percentile(widar, 25))
print(np.percentile(lenar, 75), np.percentile(widar, 75))

In [ ]:
import math

In [ ]:
np.unique(un_trash_probs)

In [ ]:
np.unique(un_other_probs)

# test flowers

In [ ]:
dirs = [d for d in TEST_SET_PATH.glob("*") if d.is_dir()]

In [ ]:
idx = 31
plt.imshow(plt.imread(dirs[idx] / "image.jpg"))

In [ ]:
edir = ExampleDir(dirs[31])

In [ ]:
neg, pos = edir.flower_neg_probs_path(), edir.flower_pos_probs_path()
neg, pos = np.load(neg), np.load(pos)

In [ ]:
idx = 0

In [ ]:
idx -= 2

In [ ]:
def trash_from_probs(trash_probs, other_probs, trash_thres=0.5, other_thres=0.85):
    has_pred = (trash_probs > 0) | (other_probs > 0)
    high_confidence_other = other_probs > other_thres
    trash_mask = trash_probs > trash_thres

    return has_pred & trash_mask & (~high_confidence_other)

In [ ]:
from mtrain.utils import overlay_mask_on_img as OV, DiskBooleanMask, DiskImage

edir = ExampleDir(dirs[idx])

# trimmed mask
mask = DiskBooleanMask.load(edir.trimmed_mask_path())

# flower
neg, pos = edir.flower_neg_probs_path(), edir.flower_pos_probs_path()
neg, pos = np.load(neg), np.load(pos)
has_flower = (pos > neg) & (pos > 0)

trash, other = edir.trash_probs_path(True), edir.other_probs_path(True)
trash, other = np.load(trash), np.load(other)
trash_mask = trash_from_probs(trash, other)
# has_trash = (trash > other) & (trash > 0)

img = plt.imread(edir.image_path)

show(
    [
        img,
        mask,
        trash,
        other,
        trash > other,
        trash_from_probs(trash, other) & (~has_flower),
    ],
    (20, 20),
    ncols=2,
    axis="off",
)
idx += 1
# show([img, mask, OV(img, mask)], (20,20), 3, "off")

In [ ]:
from tqdm import tqdm

for d in tqdm(dirs):
    ExampleDir(d).flower_pos_probs_path()
    ExampleDir(d).flower_neg_probs_path()